## Libraries

In [0]:
import os
import mlflow
import mlflow.spark

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
data = spark.read.table("workspace.telco.ml_silver_data")

In [0]:
data = data.drop("customerID")

In [0]:
display(data.limit(5))

In [0]:
feature_cols = [c for c in data.columns if c != "Churn"]


In [0]:
data = data.withColumn("label", data["Churn"].cast("double"))


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

## Modeling

In [0]:
from pyspark.ml.classification import LogisticRegression

lr_model = LogisticRegression(labelCol="label", featuresCol="features")

from pyspark.ml.classification import RandomForestClassifier

rf_model = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=200)

from pyspark.ml.classification import GBTClassifier

gb_model = GBTClassifier(labelCol="label", featuresCol="features", maxIter=50)


In [0]:
from pyspark.ml import Pipeline

pipeline1 = Pipeline(stages=[assembler, lr_model])
pipeline2 = Pipeline(stages=[assembler, rf_model])
pipeline3 = Pipeline(stages=[assembler, gb_model])

In [0]:
train, test = data.randomSplit([0.8, 0.2], seed=42)

In [0]:
fitted_lr_model = pipeline1.fit(train)
fitted_rf_model = pipeline2.fit(train)
fitted_gb_model = pipeline3.fit(train)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr_predictions = fitted_lr_model.transform(test)
rf_predictions = fitted_rf_model.transform(test)
gb_predictions = fitted_gb_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

lr_roc_auc = evaluator.evaluate(lr_predictions)
print("LogisticRegression-ROC-AUC:", lr_roc_auc)
rf_roc_auc = evaluator.evaluate(rf_predictions)
print("RandomForest-ROC-AUC:", rf_roc_auc)
gb_roc_auc = evaluator.evaluate(gb_predictions)
print("GradientBoosting-ROC-AUC:", gb_roc_auc)

